<a href="https://colab.research.google.com/github/leelemacjames/leelemacjames.github.io/blob/main/spinorDensity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
weight_audit.py — an executable audit of the spinor-density weight ledger.

Every weight in the GL(2,C) spinor-density formalism is a half-integer, so all
exponents are carried here as DOUBLED integers and printed as fractions.

For a density with residual weights (w, wbar) and index counts
(k, p) upper/lower unprimed and (l, q) upper/lower primed, the total powers of
f and fbar attaching to it under xi^A_B = f L^A_B are 2P and 2Q, with

    P = w + (k - p)/2,        Q = wbar + (l - q)/2

    W = P + Q   (dilation weight)      Delta = P - Q   (phase charge)

Run:  python3 weight_audit.py
"""

from __future__ import annotations

from dataclasses import dataclass, field
from fractions import Fraction as F
from typing import Callable, Dict, Iterable, List, Tuple


# ---------------------------------------------------------------------------
# 1. Linear forms over the rationals, so that unknown weights stay unknown.
# ---------------------------------------------------------------------------

@dataclass(frozen=True)
class Lin:
    """A linear form  const + sum_i coeff_i * var_i  with rational coefficients."""
    const: F = F(0)
    terms: Tuple[Tuple[str, F], ...] = ()

    @staticmethod
    def var(name: str) -> "Lin":
        return Lin(F(0), ((name, F(1)),))

    @staticmethod
    def num(value) -> "Lin":
        return Lin(F(value), ())

    def _as_dict(self) -> Dict[str, F]:
        d: Dict[str, F] = {}
        for k, v in self.terms:
            d[k] = d.get(k, F(0)) + v
        return {k: v for k, v in d.items() if v != 0}

    def __add__(self, other) -> "Lin":
        other = other if isinstance(other, Lin) else Lin.num(other)
        d = self._as_dict()
        for k, v in other._as_dict().items():
            d[k] = d.get(k, F(0)) + v
        return Lin(self.const + other.const,
                   tuple(sorted((k, v) for k, v in d.items() if v != 0)))

    __radd__ = __add__

    def __neg__(self) -> "Lin":
        return Lin(-self.const, tuple((k, -v) for k, v in self.terms))

    def __sub__(self, other) -> "Lin":
        other = other if isinstance(other, Lin) else Lin.num(other)
        return self + (-other)

    def __mul__(self, scalar) -> "Lin":
        s = F(scalar)
        return Lin(self.const * s, tuple((k, v * s) for k, v in self.terms))

    __rmul__ = __mul__

    def is_zero(self) -> bool:
        return self.const == 0 and not self._as_dict()

    def __str__(self) -> str:
        d = self._as_dict()
        parts: List[str] = []
        if self.const or not d:
            parts.append(str(self.const))
        for k in sorted(d):
            c = d[k]
            parts.append(f"{k}" if c == 1 else f"{c}*{k}")
        return " + ".join(parts)


def solve(equations: Iterable[Lin], unknowns: List[str]) -> Dict[str, F] | None:
    """Solve  eq = 0  for the given unknowns by elimination; None if no unique solution."""
    rows = []
    for eq in equations:
        d = eq._as_dict()
        rows.append([d.get(u, F(0)) for u in unknowns] + [-eq.const])
    n = len(unknowns)
    pivots: Dict[int, int] = {}
    r = 0
    for c in range(n):
        piv = next((i for i in range(r, len(rows)) if rows[i][c] != 0), None)
        if piv is None:
            continue
        rows[r], rows[piv] = rows[piv], rows[r]
        rows[r] = [x / rows[r][c] for x in rows[r]]
        for i in range(len(rows)):
            if i != r and rows[i][c] != 0:
                f = rows[i][c]
                rows[i] = [a - f * b for a, b in zip(rows[i], rows[r])]
        pivots[c] = r
        r += 1
    for row in rows:
        if all(x == 0 for x in row[:n]) and row[n] != 0:
            return None                      # inconsistent
    if len(pivots) < n:
        return None                          # underdetermined
    return {u: rows[pivots[c]][n] for c, u in enumerate(unknowns)}


# ---------------------------------------------------------------------------
# 2. Spinor types and their exponents.
# ---------------------------------------------------------------------------

@dataclass(frozen=True)
class SpinorType:
    """Residual weights and index counts. Weights may be numbers or Lin forms."""
    name: str
    w: object = F(0)          # residual weight
    wbar: object = F(0)       # residual anti-weight
    k: int = 0                # upper unprimed
    p: int = 0                # lower unprimed
    l: int = 0                # upper primed
    q: int = 0                # lower primed

    def _w(self) -> Lin:
        return self.w if isinstance(self.w, Lin) else Lin.num(self.w)

    def _wbar(self) -> Lin:
        return self.wbar if isinstance(self.wbar, Lin) else Lin.num(self.wbar)

    def P(self) -> Lin:
        return self._w() + Lin.num(F(self.k - self.p, 2))

    def Q(self) -> Lin:
        return self._wbar() + Lin.num(F(self.l - self.q, 2))

    def W(self) -> Lin:
        return self.P() + self.Q()

    def Delta(self) -> Lin:
        return self.P() - self.Q()


@dataclass(frozen=True)
class Charge:
    """Just the pair (W, Delta), which is all that couples to the two potentials."""
    W: Lin
    Delta: Lin

    @staticmethod
    def of(t: SpinorType) -> "Charge":
        return Charge(t.W(), t.Delta())

    def __add__(self, other: "Charge") -> "Charge":
        return Charge(self.W + other.W, self.Delta + other.Delta)

    def __neg__(self) -> "Charge":
        return Charge(-self.W, -self.Delta)

    def is_neutral(self) -> bool:
        return self.W.is_zero() and self.Delta.is_zero()

    def __str__(self) -> str:
        return f"(W = {self.W}, D = {self.Delta})"


def conjugate(t: SpinorType) -> SpinorType:
    """Conjugation exchanges primed and unprimed structure: P <-> Q."""
    return SpinorType(f"conj({t.name})", w=t.wbar, wbar=t.w,
                      k=t.l, p=t.q, l=t.k, q=t.p)


def total(*types: SpinorType) -> Charge:
    """Charge of a tensor or wedge product: charges add."""
    acc = Charge(Lin.num(0), Lin.num(0))
    for t in types:
        acc = acc + Charge.of(t)
    return acc


# ---------------------------------------------------------------------------
# 3. The tables.
# ---------------------------------------------------------------------------

EPSILONS = [
    SpinorType("eps_{AB}",   w=1,  wbar=0, p=2),
    SpinorType("eps^{AB}",   w=-1, wbar=0, k=2),
    SpinorType("eps_{A'B'}", w=0,  wbar=1, q=2),
    SpinorType("eps^{A'B'}", w=0,  wbar=-1, l=2),
]
EPS_BY_NAME = {e.name: e for e in EPSILONS}

# Consistent co-frame convention: the morphism condition is 2w+1 = 2wbar+1 = 0,
# i.e. w = wbar = -1/2 for the doubly-upper co-frame, and index position is then
# moved with the epsilons rather than by re-declaring the residual weight.
THETA_UP = SpinorType("theta^{AA'}", w=F(-1, 2), wbar=F(-1, 2), k=1, l=1)

# Inconsistent convention taken from the source appendix tables, in which the
# residual weight itself flips sign with index position.
THETA_TABLE_SOURCE = [
    SpinorType("theta_{AA'}   [src]", w=F(-1, 2), wbar=F(-1, 2), p=1, q=1),
    SpinorType("theta^A_{A'}  [src]", w=F(-1, 2), wbar=F(1, 2),  k=1, q=1),
    SpinorType("theta_A^{A'}  [src]", w=F(1, 2),  wbar=F(-1, 2), p=1, l=1),
    SpinorType("theta^{AA'}   [src]", w=F(1, 2),  wbar=F(1, 2),  k=1, l=1),
]

S_FORMS = [
    SpinorType("S^{AB}",    w=-1, wbar=0,  k=2),
    SpinorType("S^A_B",     w=0,  wbar=0,  k=1, p=1),
    SpinorType("S_{AB}",    w=1,  wbar=0,  p=2),
    SpinorType("S^{A'B'}",  w=0,  wbar=-1, l=2),
]


def lower_unprimed(t: SpinorType) -> SpinorType:
    """Contract an upper unprimed index with eps_{AB}: one upper in, one lower out."""
    assert t.k >= 1, f"{t.name} has no upper unprimed index to lower"
    e = EPS_BY_NAME["eps_{AB}"]
    return SpinorType(f"lower({t.name})",
                      w=t._w() + e._w(), wbar=t._wbar() + e._wbar(),
                      k=t.k - 1, p=t.p + 1, l=t.l, q=t.q)


def raise_unprimed(t: SpinorType) -> SpinorType:
    assert t.p >= 1, f"{t.name} has no lower unprimed index to raise"
    e = EPS_BY_NAME["eps^{AB}"]
    return SpinorType(f"raise({t.name})",
                      w=t._w() + e._w(), wbar=t._wbar() + e._wbar(),
                      k=t.k + 1, p=t.p - 1, l=t.l, q=t.q)


def lower_primed(t: SpinorType) -> SpinorType:
    assert t.l >= 1, f"{t.name} has no upper primed index to lower"
    e = EPS_BY_NAME["eps_{A'B'}"]
    return SpinorType(f"lowerPrimed({t.name})",
                      w=t._w() + e._w(), wbar=t._wbar() + e._wbar(),
                      k=t.k, p=t.p, l=t.l - 1, q=t.q + 1)


def raise_primed(t: SpinorType) -> SpinorType:
    assert t.q >= 1, f"{t.name} has no lower primed index to raise"
    e = EPS_BY_NAME["eps^{A'B'}"]
    return SpinorType(f"raisePrimed({t.name})",
                      w=t._w() + e._w(), wbar=t._wbar() + e._wbar(),
                      k=t.k, p=t.p, l=t.l + 1, q=t.q - 1)


# ---------------------------------------------------------------------------
# 4. Checks.
# ---------------------------------------------------------------------------

RESULTS: List[Tuple[bool, str, str]] = []


def check(passed: bool, title: str, detail: str = "") -> None:
    RESULTS.append((bool(passed), title, detail))


def fmt(x: Lin) -> str:
    return str(x)


def run_checks() -> None:

    # -- 1. epsilons are neutral in both channels -----------------------------
    ok = all(Charge.of(e).is_neutral() for e in EPSILONS)
    check(ok, "epsilon table: all four carry W = Delta = 0",
          "; ".join(f"{e.name}: {Charge.of(e)}" for e in EPSILONS))

    # -- 2. index movement does not change the charges ------------------------
    probe = SpinorType("psi^A", w=Lin.var("w"), wbar=Lin.var("wbar"), k=1)
    moved = lower_unprimed(probe)
    ok = (Charge.of(probe).W - Charge.of(moved).W).is_zero() and \
         (Charge.of(probe).Delta - Charge.of(moved).Delta).is_zero()
    check(ok, "lowering an index with eps preserves W and Delta (symbolic weights)",
          f"psi^A: {Charge.of(probe)}   ->   {Charge.of(moved)}")

    # -- 3. conjugation: preserves W, reverses Delta --------------------------
    c, cc = Charge.of(probe), Charge.of(conjugate(probe))
    ok = (c.W - cc.W).is_zero() and (c.Delta + cc.Delta).is_zero()
    check(ok, "conjugation preserves W and reverses Delta (derived from P <-> Q)",
          f"psi: {c}   conj(psi): {cc}")

    # -- 4. the co-frame, consistent convention -------------------------------
    th = THETA_UP
    variants = [th, lower_unprimed(th), lower_primed(th),
                lower_primed(lower_unprimed(th))]
    ok = all(Charge.of(v).is_neutral() for v in variants)
    check(ok, "co-frame at w = wbar = -1/2 carries W = Delta = 0 in every index position",
          "; ".join(f"{v.name}: {Charge.of(v)}" for v in variants))

    # -- 5. the co-frame, source-table convention: expected to FAIL -----------
    ps = [t.P() for t in THETA_TABLE_SOURCE]
    distinct = {str(p) for p in ps}
    consistent = len(distinct) == 1
    check(not consistent,
          "source co-frame table is detected as internally inconsistent (expected)",
          "; ".join(f"{t.name}: P = {fmt(t.P())}" for t in THETA_TABLE_SOURCE))

    # and the sharper form: it violates check 2 directly
    src_up = THETA_TABLE_SOURCE[3]
    src_down = THETA_TABLE_SOURCE[0]
    violates = not (Charge.of(src_up).W - Charge.of(src_down).W).is_zero()
    check(violates,
          "source table violates index-movement invariance (expected)",
          f"W(theta^{{AA'}}) = {fmt(Charge.of(src_up).W)} vs "
          f"W(theta_{{AA'}}) = {fmt(Charge.of(src_down).W)}, "
          "but the epsilons that move the indices are neutral")

    # -- 6. S-forms are mutually consistent -----------------------------------
    ok = all(Charge.of(s).is_neutral() for s in S_FORMS)
    check(ok, "S-form table: all four carry W = Delta = 0",
          "; ".join(f"{s.name}: {Charge.of(s)}" for s in S_FORMS))

    # Sigma^A_B ~ theta^A_{A'} ^ theta^{BA'} must match S^A_B
    th_mixed = lower_primed(THETA_UP)          # theta^A_{A'}, derived with eps
    th_up2 = THETA_UP                           # theta^{BA'}
    sigma = total(th_mixed, th_up2)
    ok = sigma.is_neutral()
    check(ok, "Sigma^A_B built from two co-frames agrees with the tabulated S^A_B",
          f"theta ^ theta: {sigma}")

    # -- 7. sesquilinear no-go ------------------------------------------------
    psi = SpinorType("psi", w=Lin.var("v"), wbar=Lin.var("v"))   # W = 2v, Delta = 0
    pair = total(conjugate(psi), psi)
    ok = (pair.W - Charge.of(psi).W * 2).is_zero() and pair.Delta.is_zero()
    check(ok, "sesquilinear pairing doubles W and is automatically phase-neutral",
          f"conj(psi) (x) psi: {pair}")

    sol = solve([pair.W], ["v"])
    ok = sol is not None and sol["v"] == 0
    check(ok, "invariance of a sesquilinear term with neutral measure forces W = 0",
          f"solving W(conj(psi) (x) psi) = 0 gives v = {sol['v'] if sol else 'no unique solution'}")

    # -- 8. dual pairing, arbitrary magnitude ---------------------------------
    alpha = SpinorType("alpha^A", w=Lin.var("nu_a"), wbar=Lin.var("nu_b"), k=1)
    beta = SpinorType("beta_A",
                      w=-Lin.var("nu_a"), wbar=-Lin.var("nu_b"), p=1)
    pairing = total(alpha, beta)
    ok = pairing.is_neutral()
    check(ok, "dual pairing alpha^A ^ beta_A is neutral for arbitrary weights",
          f"alpha: {Charge.of(alpha)}   beta: {Charge.of(beta)}   product: {pairing}")

    # the special case of vanishing residual weights
    a0 = SpinorType("alpha^A", k=1)
    b0 = SpinorType("beta_A", p=1)
    ok = (Charge.of(a0).W - Lin.num(F(1, 2))).is_zero() and \
         (Charge.of(b0).W + Lin.num(F(1, 2))).is_zero() and \
         total(a0, b0).is_neutral()
    check(ok, "at vanishing residual weight the dyad sits at W = +1/2 and -1/2",
          f"alpha: {Charge.of(a0)}   beta: {Charge.of(b0)}")

    # -- 9. duality against conjugation ---------------------------------------
    x = SpinorType("x", w=Lin.var("P"), wbar=Lin.var("Q"))
    dual_x = -Charge.of(x)
    conj_x = Charge.of(conjugate(x))
    sol = solve([dual_x.W - conj_x.W], ["P", "Q"])
    # W(dual) = -W, W(conj) = +W, so the equation is 2W = 0, i.e. P + Q = 0
    eq = dual_x.W - conj_x.W
    forced = solve([eq, Lin.var("P") - Lin.var("Q")], ["P", "Q"])
    ok = forced is not None and forced["P"] + forced["Q"] == 0
    check(ok, "dual = conjugate forces W = 0 (the conditional obstruction)",
          f"equation: {fmt(eq)} = 0, i.e. W = 0; Delta is left free")

    # the phase channel is untouched by the same identification
    eq_phase = dual_x.Delta - conj_x.Delta
    check(eq_phase.is_zero(),
          "the same identification places no condition on Delta",
          f"Delta(dual) - Delta(conj) = {fmt(eq_phase)}")

    # -- 10. the scalar sector L_H --------------------------------------------
    # sqrt(g) and g^{mu nu} are neutral once the co-frame is (checks 4, 6).
    phi = SpinorType("phi", w=Lin.var("v_phi"), wbar=Lin.var("v_phi"))
    pi = SpinorType("pi", w=Lin.var("v_pi"), wbar=Lin.var("v_pi"))
    t1 = total(conjugate(pi), phi)                      # pibar D phi
    t2 = total(conjugate(pi), pi)                       # pibar pi g
    sol = solve([t1.W, t2.W], ["v_phi", "v_pi"])
    ok = sol is not None and sol["v_phi"] == 0 and sol["v_pi"] == 0
    check(ok, "scalar sector: invariance of every term of L_H forces W(phi) = 0",
          f"terms give W = {fmt(t1.W)} and {fmt(t2.W)}; solution {sol}")

    # -- 11. the fermionic sector ---------------------------------------------
    # S ^ rho ^ D lambda with rho = theta lambda-tilde, S and theta neutral.
    lam = SpinorType("lambda_A", w=Lin.var("v_lam"), wbar=Lin.var("v_lam"), p=1)
    rho = total(THETA_UP, conjugate(lam))               # rho^C = theta^{CC'} lam~_{C'}
    S = S_FORMS[1]                                      # S^A_B, neutral
    term = Charge.of(S) + rho + Charge.of(lam)
    sol = solve([term.W], ["v_lam"])
    W_lam = Charge.of(lam).W
    ok = sol is not None and (W_lam.const + sum(c * sol[k] for k, c in W_lam.terms)) == 0
    val = None if sol is None else W_lam.const + sum(c * sol[k] for k, c in W_lam.terms)
    check(ok, "fermionic sector: invariance of the matter term forces W(lambda) = 0",
          f"term charge W = {fmt(term.W)} where W(lambda) = {fmt(W_lam)}; "
          f"solving gives W(lambda) = {val}")


# ---------------------------------------------------------------------------
# 5. Report.
# ---------------------------------------------------------------------------

def main() -> int:
    run_checks()
    width = 78
    print("=" * width)
    print("SPINOR-DENSITY WEIGHT LEDGER — AUDIT")
    print("=" * width)
    failures = 0
    for i, (passed, title, detail) in enumerate(RESULTS, 1):
        mark = "PASS" if passed else "FAIL"
        if not passed:
            failures += 1
        print(f"\n[{mark}] {i:2d}. {title}")
        if detail:
            for line in _wrap(detail, width - 6):
                print(f"      {line}")
    print("\n" + "=" * width)
    print(f"{len(RESULTS) - failures} of {len(RESULTS)} checks passed.")
    print("=" * width)
    return 0 if failures == 0 else 1


def _wrap(text: str, width: int) -> List[str]:
    out, line = [], ""
    for word in text.split(" "):
        if len(line) + len(word) + 1 > width:
            out.append(line)
            line = word
        else:
            line = f"{line} {word}".strip()
    if line:
        out.append(line)
    return out


# if __name__ == "__main__":
#     raise SystemExit(main())

if __name__ == "__main__":
    import sys
    code = main()
    if "ipykernel" not in sys.modules:
        raise SystemExit(code)
# if __name__ == "__main__":
#     raise SystemExit(main())

SPINOR-DENSITY WEIGHT LEDGER — AUDIT

[PASS]  1. epsilon table: all four carry W = Delta = 0
      eps_{AB}: (W = 0, D = 0); eps^{AB}: (W = 0, D = 0); eps_{A'B'}: (W = 0,
      D = 0); eps^{A'B'}: (W = 0, D = 0)

[PASS]  2. lowering an index with eps preserves W and Delta (symbolic weights)
      psi^A: (W = 1/2 + w + wbar, D = 1/2 + w + -1*wbar) -> (W = 1/2 + w +
      wbar, D = 1/2 + w + -1*wbar)

[PASS]  3. conjugation preserves W and reverses Delta (derived from P <-> Q)
      psi: (W = 1/2 + w + wbar, D = 1/2 + w + -1*wbar) conj(psi): (W = 1/2 + w
      + wbar, D = -1/2 + -1*w + wbar)

[PASS]  4. co-frame at w = wbar = -1/2 carries W = Delta = 0 in every index position
      theta^{AA'}: (W = 0, D = 0); lower(theta^{AA'}): (W = 0, D = 0);
      lowerPrimed(theta^{AA'}): (W = 0, D = 0);
      lowerPrimed(lower(theta^{AA'})): (W = 0, D = 0)

[PASS]  5. source co-frame table is detected as internally inconsistent (expected)
      theta_{AA'} [src]: P = -1; theta^A_{A'} [src]: P = 0; 